# Bestand der Personenwagen nach Marke, Stadtquartier und Jahr, seit 2002
Datum: 07.02.2025

Dieser Datensatz beinhaltet die Anzahl Personenwagen der Stadt Zürich nach Automarke, Stadtquartier und Jahr seit 2002.

**Dataset auf PROD Datenkatalog**:  https://data.stadt-zuerich.ch/dataset/prd_ssz_fz_pw_bestand_marke_quartier_od2003

### Colab
Mit Colab kann das Jupyter-Notebook interaktiv im Browser gestartet werden. 

Klicke auf den Button:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DonGoginho/colab/blob/main/pw/od2003_prd_ssz_fz_pw_bestand_marke_quartier_od2003.ipynb)



### Importiere die notwendigen Packages

In [1]:
#%pip install altair datetime folium geopandas io requests matplotlib numpy pandas seaborn plotly
!pip install altair==5.0.1 vl-convert-python

In [6]:
import altair as alt
import datetime
import folium 
import geopandas as gpd
import io
from IPython.display import Markdown as md
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
#import pivottablejs
#from pivottablejs import pivot_ui
import plotly.express as px
import requests
import seaborn as sns

Importiere die eigenen Funktionen, die unter ../0_scripts abegelegt sind:

1) Klone das Github-Repo auf Colab. Sonst werden die Skripts nicht gelesen...

In [7]:
!git clone https://github.com/DonGoginho/colab.git


fatal: destination path 'colab' already exists and is not an empty directory.


2) Checke die Schreibrechte in den geklonten Verzeichnissen

In [8]:
!ls -l /content/colab
!ls -l /content/colab/dogs


Der Befehl "ls" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.
Der Befehl "ls" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.


3) Importiere die Skripts

In [ ]:
import sys
sys.path.append('/content/colab/0_scripts')

import my_py_dataviz_functions as mypy_dv
import my_py_dataloading_functions as mypy_dl

In [ ]:
#help(mypy_dv)
#help(mypy_dl)

In [ ]:
SSL_VERIFY = False
# evtl. SSL_VERIFY auf False setzen wenn die Verbindung zu https://www.gemeinderat-zuerich.ch nicht klappt (z.B. wegen Proxy)
# Um die SSL Verifikation auszustellen, bitte die nächste Zeile einkommentieren ("#" entfernen)
# SSL_VERIFY = False

In [ ]:
if not SSL_VERIFY:
    import urllib3
    urllib3.disable_warnings()

### Settings
Definiere Settings. 
Hier das Zahlenformat von Float-Werten (z.B. *'{:,.2f}'.format* mit Komma als Tausenderzeichen)

In [ ]:
#pd.options.display.float_format = lambda x : '{:,.1f}'.format(x) if (np.isnan(x) | np.isinf(x)) else '{:,.0f}'.format(x) if int(x) == x else '{:,.1f}'.format(x)
pd.options.display.float_format = '{:.0f}'.format
pd.set_option('display.width', 100)
pd.set_option('display.max_columns', 15)

#### Zeitvariabeln


In [ ]:
#Zeitvariabeln als Strings:
now = datetime.date.today()
year_today = now.strftime("%Y")
date_today = "_"+now.strftime("%Y-%m-%d")

#Zeitvariabeln als Integers:
int_times = now.timetuple()
aktuellesJahr = int_times[0]
aktuellerMonat = int_times[1]
selectedMonat = int_times[1]-2
#print(aktuellesJahr, aktuellerMonat,'datenstand: ', selectedMonat, int_times)

### Daten importieren und Metadaten checken

- status: int / prod
- data_source: web / ld / dropzone
- datums_attr: beschreibt das oder die Datumsfelder, die als Datum geparsed werden sollen. Falls es keines gibt: None

In [ ]:
package_name = "prd_ssz_fz_pw_bestand_marke_quartier_od2003"

In [ ]:
data2betested = mypy_dl.load_data(
    status = 'prod'
    , data_source = 'web'
    , package_name = package_name
    , dataset_name = "VER200OD2003"      
    , datums_attr = ['StichtagDatJahr']
    )

In [ ]:
data2betested.head(4).T

Berechne weitere Attribute falls notwendig

In [ ]:
data2betested = (
    data2betested
    .copy()
    .assign(
        #Aktualisierungs_Datum_str= lambda x: x.Aktualisierungs_Datum.astype(str),
        StichtagDatJahr_str = lambda x: x.StichtagDatJahr.astype(str),
        Jahr = lambda x: x.StichtagDatJahr,
        Jahr_end = lambda x: x.StichtagDatJahr+pd.offsets.YearEnd(0),
        Jahr_nbr = lambda x: x.Jahr.dt.year,
    )
    .sort_values('StichtagDatJahr', ascending=False)
    )
data2betested.dtypes

In [ ]:
data2betested.head(2)

Minimales und maximales Jahr im Datensatz

In [ ]:
data_max_date = str(max(data2betested.Jahr).year)
data_min_date = str(min(data2betested.Jahr).year)

print(f"Die Daten haben ein Minimumjahr von {data_min_date} und ein Maximumjahr von {data_max_date}")

### Verwende das Datum als Index

While we did already parse the `datetime` column into the respective datetime type, it currently is just a regular column. 
**To enable quick and convenient queries and aggregations, we need to turn it into the index of the DataFrame**

In [ ]:
data2betested = data2betested.set_index("StichtagDatJahr") 
data2betested = data2betested.sort_index()

In [ ]:
data2betested.index.year.unique()

### Einfache Datentests

In [ ]:
data2betested.info(memory_usage='deep', verbose=True)

In [ ]:
print(f'The dataset has {data2betested.shape[0]:,.0f} rows (observations) and {data2betested.shape[1]:,.0f} columns (variables).')
print(f'There seem to be {data2betested.duplicated().sum()} exact duplicates in the data.')

### Pivotiere

In [ ]:
pivoted_df = data2betested.pivot_table(
    index='Jahr',
    columns='FzMarkeLang',
    values='FzAnz',
    aggfunc='sum'
)

pivoted_df = pivoted_df.sort_values(by=pivoted_df.index[-1], axis=1, ascending=False)

# Sortieren der Zeilen (Jahre) in absteigender Reihenfolge
pivoted_df = pivoted_df.sort_index(ascending=False)

# Anzeigen des Ergebnisses
pivoted_df.head(8).T

In [ ]:
pivoted_df_diff = pivoted_df.sort_index(ascending=True).diff()
pivoted_df_diff.tail(5)

### Top Marken des aktuellen Jahrs

In [ ]:
agg_marken = data2betested.loc[data_min_date:data_max_date]\
    .groupby(['FzMarkeLang']) \
    .agg(sum_FzAnz=('FzAnz', 'sum')) \
    .sort_values('FzMarkeLang', ascending=False)

agg_marken.reset_index().query('sum_FzAnz >3500').sort_values('FzMarkeLang', ascending=False)

In [ ]:
#    .query('FzMarkeLang!="Rover"')\
agg_marke_years = data2betested\
    .groupby(['StichtagDatJahr','Jahr_nbr','FzMarkeLang']) \
    .agg(sum_FzAnz=('FzAnz', 'sum')) \
    .sort_values(['StichtagDatJahr','sum_FzAnz'], ascending=[False, False]) 
agg_marke_years.reset_index().head(5)

Mach eine Liste mit den Top15 Fahrzeugen im aktuellen Jahr

In [ ]:
df_top_25 = agg_marke_years.reset_index().query('StichtagDatJahr == '+data_max_date).sort_values('sum_FzAnz', ascending=False).head(25)
df_top_5 = agg_marke_years.reset_index().query('StichtagDatJahr == '+data_max_date).sort_values('sum_FzAnz', ascending=False).head(5)
df_top_25.head(2)
#df_list_top_15_23 = top_15[['FzMarkeLang','sum_FzAnz']]
list_top_25_latest_year = df_top_25['FzMarkeLang'].tolist()
list_top_5_latest_year = df_top_5['FzMarkeLang'].tolist()

#list_top_25_latest_year


In [ ]:
#list_top_25_latest_year

Wähle aus dem nach Jahr aggregierten Dataframe nur jene aus der Top25-Liste aus

In [ ]:
#agg_marke_years.reset_index()

In [ ]:
df_sel_top25_since2002 = agg_marke_years.reset_index().query('FzMarkeLang in @list_top_25_latest_year')
df_sel_top5_since2002 = agg_marke_years.reset_index().query('FzMarkeLang in @list_top_5_latest_year')
#df_sel_top25_since2002

In [ ]:
df_sel_top25_since2002.head(2)

In [ ]:
grafik1 = mypy_dv.plot_altair_multiline_highlight(
    data = df_sel_top25_since2002.reset_index().sort_values('FzMarkeLang', ascending=True)
    ,x = 'StichtagDatJahr:T'
    ,y = 'sum_FzAnz:Q'
    ,x_beschriftung = 'Jahr'
    , y_beschriftung = 'Anz. Personen'
    ,category = "FzMarkeLang:N"
    ,category_beschriftung= 'Legende:'
    ,warning_status = "ignore" #always or ignore
    ,myTitle="Entwicklung der Top-25 Marken von "+data_max_date+", seit "+data_min_date
)
grafik1

#### Faced Grids

In [ ]:
data2betested.columns


In [ ]:
myFG = data2betested.loc[data_min_date:data_max_date]\
    .query('FzMarkeLang in @list_top_5_latest_year')\
    .groupby(['StichtagDatJahr', 'KreisLang', 'KreisCd', 'KreisSort','FzMarkeLang']) \
    .agg(sum_FzAnz=('FzAnz', 'sum')) \
    .sort_values('KreisSort', ascending=True) 

myFG.reset_index().head(3)

In [ ]:
faced_grid = mypy_dv.plot_sns_facetgrid(
    data = myFG.reset_index().sort_values('KreisSort', ascending=True)
    ,col = "KreisLang"
    ,hue = "FzMarkeLang"
    ,col_wrap = 4
    ,grafiktyp = sns.lineplot
    ,x = "StichtagDatJahr"
    ,y = "sum_FzAnz"
    ,ylabel= "Anzahl Fahrzeuge"
    ,warning_status ="ignore"
    ,height = 3
    ,myTitle="Entwicklung der Top-5 Marken von "+data_max_date+' seit '+data_min_date
)
faced_grid

In [ ]:
myFG1 = data2betested.loc[data_min_date:data_max_date]\
    .query('FzMarkeLang in @list_top_25_latest_year')\
    .groupby(['StichtagDatJahr', 'KreisLang', 'KreisCd', 'KreisSort','FzMarkeLang']) \
    .agg(sum_FzAnz=('FzAnz', 'sum')) \
    .sort_values('KreisSort', ascending=True) 

myFG1.reset_index().head(3)

In [ ]:
faced_grid1 = mypy_dv.plot_sns_facetgrid(
    data = myFG1.reset_index().sort_values('KreisSort', ascending=True)
    ,col = "FzMarkeLang"
    ,hue = "KreisLang"
    ,col_wrap = 5
    ,grafiktyp = sns.lineplot
    ,x = "StichtagDatJahr"
    ,y = "sum_FzAnz"
    ,ylabel= "Anzahl Fahrzeuge"
    ,warning_status ="ignore"
    ,height = 3
    ,myTitle="Entwicklung der Top-5 Marken von "+data_max_date+' seit '+data_min_date
)
faced_grid1

In [ ]:
myFG3 = data2betested.loc[data_min_date:data_max_date]\
    .query('FzMarkeLang in @list_top_25_latest_year')\
    .groupby(['StichtagDatJahr', 'Jahr_nbr','FzMarkeLang']) \
    .agg(sum_FzAnz=('FzAnz', 'sum')) \
    .sort_values('Jahr_nbr', ascending=True) 

myFG3.reset_index().head(3)

In [ ]:
# Beispiel-Daten
data = myFG3.reset_index().sort_values(['Jahr_nbr','sum_FzAnz'], ascending=[False, False])

# Reihenfolge der Jahre festlegen (sortiert)
jahr_order = sorted(data['Jahr_nbr'].unique())
#print(jahr_order)

# FacetGrid erstellen
facet_grid = sns.FacetGrid(
    data=data,
    col="FzMarkeLang",
    hue="FzMarkeLang",
    col_wrap=5,
    height=3,
    palette="Set1"
)

# Barplot auf jedes Facet anwenden
facet_grid.map(
    sns.barplot,
    "Jahr_nbr",  # x-Achse
    "sum_FzAnz", # y-Achse
    order=jahr_order  
)

# Titel und Achsenbeschriftungen hinzufügen
facet_grid.set_axis_labels("Jahr", "Anzahl Fahrzeuge")
facet_grid.set_titles("{col_name}")
facet_grid.set_xticklabels(rotation=45)
# Nach facet_grid.map()

#facet_grid.add_legend()

# Layout anpassen und anzeigen
plt.subplots_adjust(top=0.9)
facet_grid.fig.suptitle("Entwicklung der Top-25 Marken im " + data_max_date + ' seit ' + data_min_date)

plt.show()

#### Treemaps

**Funktion zum einfärben**

*Comment: Muss ich noch als Funktion umsetzen*

In [ ]:
  qual12br = ["#5D4BFE", "#4AA9FF", "#55FFFF", "#986AD5", "#FC4C99", "#FF919A", "#349894", "#44B14A", "#B7E14E", "#B97624", "#FF7231", "#FFD736"]
  qual12 = ["#3431DE", "#0A8DF6", "#23C3F1", "#7B4FB7", "#DB247D", "#FB737E", "#007C78", "#1F9E31", "#99C32E", "#9A5B01", "#FF720C", "#FBB900"]
  qual12da = ["#0017BF", "#0072D7", "#00A5D2", "#5E359A", "#BA0062", "#DA5563", "#00615D", "#00770F", "#7BA600", "#7B4100", "#DC5500", "#DA9C00"]
  div9val = ["#782600", "#CC4309", "#FF720C", "#FFBC88", "#E4E0DF", "#AECBFF", "#6B8EFF", "#3B51FF", "#2F2ABB",]
  div9ntr = ["#A30059", "#DB247D", "#FF579E", "#FFA8D0", "#E4E0DF", "#A8DBB1", "#55BC5D", "#1F9E31", "#10652A",]

In [ ]:
df_top_35_alltime = agg_marken.reset_index().sort_values('sum_FzAnz', ascending=False).head(25)
df_top_5_alltime  = agg_marken.reset_index().sort_values('sum_FzAnz', ascending=False).head(5)
df_top_35_alltime.head(2)
#df_list_top_15_23 = top_15[['FzMarkeLang','sum_FzAnz']]
list_top_35_alltime = df_top_35_alltime['FzMarkeLang'].tolist()
list_top_5_alltime = df_top_5_alltime['FzMarkeLang'].tolist()

In [ ]:
# Extrahiere die Top-Fahrzeugmarken
#attr2becolored = data2betested['FzMarkeLang'].unique().tolist()
attr2becolored = list_top_35_alltime
# Verfügbare Farben
verfügbare_farben_zuericolors = qual12da+qual12br+qual12+div9ntr

# Erstelle das Farben-Dictionary
farben_dict_zc = {'(?)':'lightgrey'}
for index, x in enumerate(attr2becolored):
    farben_dict_zc[x] = verfügbare_farben_zuericolors[index % len(verfügbare_farben_zuericolors)]

# Das resultierende Farben-Dictionary
print(farben_dict_zc)
#print(verfügbare_farben_zuericolors)

In [ ]:
myTreemapAgg = data2betested.loc[data_max_date]\
    .groupby(['StichtagDatJahr', 'FzMarkeLang','QuarLang', 'QuarCd', 'QuarSort', 'KreisLang', 'KreisCd', 'KreisSort']) \
    .agg(sum_FzAnz=('FzAnz', 'sum')) \
    .sort_values('sum_FzAnz', ascending=False) 

##### Fahrzeugmarken nach Kreis, Quartier

In [ ]:
#data=data2betested.loc[(data2betested.index.year == 2004)| (data2betested.index.year == 2014) | (data2betested.index.year == 2024)].query("FzAnz>50")
treeMap0 = mypy_dv.plot_px_treemap(
    data=data2betested.loc[(data2betested.index.year == 2004)| (data2betested.index.year == 2024)].query("FzAnz>50")
    ,levels=['Jahr_nbr', 'FzMarkeLang','KreisLang', 'QuarLang']
    ,values="FzAnz"
    ,color="FzMarkeLang"
    #, color_continuous_scale='Blues'
    ,color_discrete_map=farben_dict_zc
    ,height=600
    ,width=1100               
    #,margin_val_bottom=25
    ,myHeaderTitle="Fahrzeugbestände nach Marke, Stadtkreis und Stadtquartier, 2004 vs "+data_max_date
)
treeMap0

In [ ]:
treeMap1 = mypy_dv.plot_px_treemap(
    data=data2betested.loc[(data2betested.index.year == 2004)| (data2betested.index.year == 2014) | (data2betested.index.year == 2024)].query("FzAnz>20")
    ,levels=['QuarLang','FzMarkeLang','Jahr_nbr', ]
    ,values="FzAnz"
    ,color="FzMarkeLang"
    #, color_continuous_scale='Blues'
    ,color_discrete_map=farben_dict_zc
    ,height=600
    ,width=1100               
    #,margin_val_bottom=25
    ,myHeaderTitle="Fahrzeugbestände nach Stadtquartier, Marke und Jahr (2004, 2014 und "+data_max_date +")"
)
treeMap1